# Video Indexing at Speed - Kaggle Execution Notebook

### Instructions:
1. **Settings -> Accelerator**: Set to **GPU T4 x 2** (Dual T4 for 2-GPU sharding) or **GPU P100**.
2. **Settings -> Internet**: Toggle **ON**.
3. **HuggingFace Access**: Accept dataset terms at [huggingface.co/datasets/HuggingFaceFV/finevideo](https://huggingface.co/datasets/HuggingFaceFV/finevideo) and create a Read Token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

In [ ]:
# Step 1: Inspect Kaggle Hardware Specs for HARDWARE.md
!nvidia-smi
!lscpu | grep -E "Model name|CPU\(s\):"
!free -h
!df -h /kaggle/working

In [ ]:
# Step 2: Install Python Requirements
!pip install --upgrade pip
!pip install -r requirements.txt

In [ ]:
# Step 3: Run Pure-Python Unit Tests
!python -m unittest discover -s tests -v

In [ ]:
# Step 4: Preparation Phase (download weights & warmup)
# Standard preparation:
!python run.py --prepare --batch-size 64

# Optional: If TensorRT is available in your environment, build FP16 engine:
# !python run.py --prepare --export-trt --batch-size 64

In [ ]:
# Step 5: Authenticate with HuggingFace and Download 50 Parquet Files
# Paste your HuggingFace read token below:
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN", "PASTE_YOUR_HF_READ_TOKEN_HERE")
login(token=hf_token)

# Download 50 files (~25 GB) directly to /kaggle/working/finevideo/
!hf download HuggingFaceFV/finevideo --repo-type dataset --include "data/train-000[0-4]*" --local-dir finevideo

In [ ]:
# Step 6: Baseline Run on 20 Videos (Verify accuracy, speed, and output format)
!python run.py --input finevideo/data --output results_baseline.jsonl --limit 20 --fps 5 --batch-size 64
!head -n 5 results_baseline.jsonl
!tail -n 5 results_baseline.jsonl

In [ ]:
# Step 7: Full 50-File Run (Multi-GPU Sharding on Dual T4, or Single GPU)
import time, torch

start_time = time.time()
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"[Kaggle] Starting full run with {num_gpus} GPU(s) at: {time.strftime('%Y-%m-%d %H:%M:%S')}")

if num_gpus >= 2:
    print("[Kaggle] Launching 2 parallel shards across cuda:0 and cuda:1...")
    !python run.py --input finevideo/data --output results.jsonl --shard-index 0 --num-shards 2 --device cuda:0 --workers 6 --fps 5 --batch-size 64 &
    !python run.py --input finevideo/data --output results.jsonl --shard-index 1 --num-shards 2 --device cuda:1 --workers 6 --fps 5 --batch-size 64 &
    # Note: in Jupyter cell, run via bash script or subshell:
    !bash -c "python run.py --input finevideo/data --output results.jsonl --shard-index 0 --num-shards 2 --device cuda:0 --workers 6 --fps 5 --batch-size 64 & python run.py --input finevideo/data --output results.jsonl --shard-index 1 --num-shards 2 --device cuda:1 --workers 6 --fps 5 --batch-size 64 & wait && cat results.shard0of2.jsonl results.shard1of2.jsonl > results.jsonl"
else:
    print("[Kaggle] Launching single GPU run...")
    !python run.py --input finevideo/data --output results.jsonl --fps 5 --batch-size 64

elapsed = time.time() - start_time
print(f"[Kaggle] Finished full run in {elapsed:.2f} seconds ({elapsed / 3600:.2f} hours)")

In [ ]:
# Step 8: Compress Output to results.jsonl.gz and Check Integrity
!gzip -k -9 -f results.jsonl
!ls -lh results.jsonl results.jsonl.gz
!wc -l results.jsonl
!head -n 5 results.jsonl
!tail -n 5 results.jsonl